# 02 · NumPy: คำนวณข้อมูลตัวเลขและกริดเชิงพื้นที่
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/02_NumPy_for_Spatial_Data.ipynb)

**เวลา:** 2 ชั่วโมง | **ระดับ:** 🟢 ง่าย → 🟡 ปานกลาง → 🔴 ยาก

**ทำไมนักภูมิศาสตร์ต้องรู้ NumPy?** ข้อมูล Raster (DEM, ภาพดาวเทียม, กริดฝน) คือ *ตารางตัวเลข 2 มิติ* ซึ่ง NumPy จัดการได้เร็วกว่า list หลายร้อยเท่า

```
list  →  วนทีละค่า (ช้า)
array →  คำนวณทั้งชุดพร้อมกัน (vectorization, เร็ว)
```

## 🟢 1. สร้าง array และคำนวณทั้งชุดพร้อมกัน

In [ ]:
import numpy as np

rain = np.array([5.2, 8.1, 20.4, 55.0, 160.3, 140.8, 180.2, 235.6, 210.4, 105.7, 30.2, 7.5])
months = np.array(["ม.ค.","ก.พ.","มี.ค.","เม.ย.","พ.ค.","มิ.ย.","ก.ค.","ส.ค.","ก.ย.","ต.ค.","พ.ย.","ธ.ค."])

print("ชนิด:", type(rain), "| รูปร่าง (shape):", rain.shape)
print("แปลงเป็นนิ้วทั้งชุด:", np.round(rain / 25.4, 2))     # ไม่ต้องใช้ loop!

In [ ]:
# ฟังก์ชันสถิติพื้นฐาน
print(f"รวม   = {rain.sum():.1f} มม.")
print(f"เฉลี่ย = {rain.mean():.1f} มม.")
print(f"SD    = {rain.std(ddof=1):.1f} มม.")        # ddof=1 → SD ของกลุ่มตัวอย่าง
print(f"มากสุด = {rain.max()} มม. เดือน {months[rain.argmax()]}")   # argmax = ตำแหน่งของค่ามากสุด
print(f"ร้อยละของฝนแต่ละเดือนต่อทั้งปี:\n{np.round(rain/rain.sum()*100, 1)}")

## 🟢 2. Boolean mask: คัดกรองข้อมูลตามเงื่อนไข
แนวคิดเดียวกับ *Select by Attribute* ใน GIS

In [ ]:
dry_mask = rain < 50              # ได้ array ของ True/False
print(dry_mask)
print("เดือนแล้ง (ฝน < 50 มม.):", months[dry_mask])
print("จำนวนเดือนแล้ง:", dry_mask.sum())          # True = 1, False = 0

wet = (rain >= 100) & (rain <= 200)                 # และ = &   หรือ = |   (ต้องมีวงเล็บ)
print("เดือนที่ฝน 100–200 มม.:", months[wet])

## 🟡 3. Array 2 มิติ: หลายสถานี × หลายเดือน
`axis=0` = คำนวณตามแนวตั้ง (ข้ามแถว), `axis=1` = คำนวณตามแนวนอน (ข้ามคอลัมน์)

In [ ]:
# แถว = สถานี, คอลัมน์ = เดือน
names = np.array(["เชียงใหม่", "น่าน", "แม่ฮ่องสอน"])
R = np.array([
    [5, 8, 20, 55, 160, 140, 180, 235, 210, 105, 30, 7],
    [8, 12, 30, 80, 190, 170, 230, 290, 240, 110, 35, 10],
    [4, 6, 15, 45, 150, 160, 190, 210, 180,  95, 25, 6],
])
print("shape =", R.shape, "→", R.shape[0], "สถานี ×", R.shape[1], "เดือน")

annual = R.sum(axis=1)            # รวมรายปีของแต่ละสถานี
monthly_mean = R.mean(axis=0)     # ค่าเฉลี่ยของทุกสถานีในแต่ละเดือน
for n, a in zip(names, annual):
    print(f"{n:12s} ฝนรายปี {a:5d} มม.")
print("ค่าเฉลี่ยรายเดือนของภูมิภาค:", np.round(monthly_mean, 1))
print("สถานีที่ฝนมากที่สุด:", names[annual.argmax()])

In [ ]:
# Anomaly (ค่าผิดปกติ) = ค่าจริง − ค่าเฉลี่ย  → broadcasting: NumPy ขยายขนาดให้อัตโนมัติ
anomaly = R - monthly_mean          # (3,12) − (12,) ได้เลย
print(np.round(anomaly, 1))

## 🟡 4. สร้างกริด DEM จำลองด้วย `np.meshgrid`
Raster = กริดของค่า Z บนพิกัด X, Y

In [ ]:
import matplotlib.pyplot as plt

# พื้นที่ 20 × 20 กม. ความละเอียด 100 ม. (ช่องกริด 200 × 200)
x = np.arange(0, 20000, 100)        # พิกัด x (เมตร)
y = np.arange(0, 20000, 100)
X, Y = np.meshgrid(x, y)            # สร้างพิกัดทุกช่องกริด

# ภูเขาจำลอง 2 ลูก ด้วยฟังก์ชัน Gaussian + พื้นราบสูง 300 ม.
def hill(X, Y, x0, y0, h, s):
    return h * np.exp(-((X-x0)**2 + (Y-y0)**2) / (2*s**2))

DEM = 300 + hill(X, Y, 6000, 12000, 1400, 3000) + hill(X, Y, 14000, 6000, 900, 2500)
print("shape:", DEM.shape, "| ต่ำสุด %.0f ม. | สูงสุด %.0f ม." % (DEM.min(), DEM.max()))

plt.imshow(DEM, origin="lower", extent=[0, 20, 0, 20], cmap="terrain")
plt.colorbar(label="Elevation (m)")
plt.title("Synthetic DEM"); plt.xlabel("x (km)"); plt.ylabel("y (km)")
plt.show()

## 🔴 5. วิเคราะห์ภูมิประเทศจาก DEM: อุณหภูมิ และความลาดชัน (Slope)
$$slope = \arctan\sqrt{\left(\frac{\partial z}{\partial x}\right)^2+\left(\frac{\partial z}{\partial y}\right)^2}$$

In [ ]:
# 5.1 อุณหภูมิจาก lapse rate: คำนวณทั้งกริด 40,000 ช่องในบรรทัดเดียว
T = 30 - 6.5 * DEM / 1000

# 5.2 ความลาดชัน: np.gradient คืนค่าอนุพันธ์ตามแกน y และ x (ระยะห่างกริด 100 ม.)
dz_dy, dz_dx = np.gradient(DEM, 100, 100)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

# 5.3 Reclassify: พื้นที่ลาดชันเกิน 30° (เสี่ยงดินถล่ม)
risk = slope_deg > 30
cell_area_km2 = (100 * 100) / 1e6
print(f"ความชันสูงสุด {slope_deg.max():.1f}° | พื้นที่ชัน > 30° = {risk.sum()*cell_area_km2:.2f} ตร.กม.")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, data, cm, lab in zip(ax, [T, slope_deg, risk], ["coolwarm", "YlOrRd", "Reds"],
                            ["Temperature (°C)", "Slope (°)", "Slope > 30°"]):
    im = a.imshow(data, origin="lower", extent=[0, 20, 0, 20], cmap=cm)
    fig.colorbar(im, ax=a, label=lab); a.set_title(lab)
plt.tight_layout(); plt.show()

---
## ✏️ แบบฝึกหัด
**🟢** 1) หาค่ามัธยฐาน (`np.median`) และพิสัย (`max−min`) ของ `rain`  2) แปลงอุณหภูมิ °C → K ของ array `T`

**🟡** 3) จาก `R` หาว่าแต่ละสถานีมีกี่เดือนที่ฝน ≥ 100 มม. (ใช้ mask + `sum(axis=1)`)
4) คำนวณ *ดัชนีความเข้มข้นของฝนตามฤดู* = ฝน พ.ค.–ต.ค. ÷ ฝนทั้งปี ของแต่ละสถานี

**🔴** 5) คำนวณ **Aspect** (ทิศด้านลาด) จาก `dz_dx`, `dz_dy` ด้วย `np.arctan2` แล้วแสดงแผนที่
6) หาพื้นที่ (ตร.กม.) ที่ *สูงกว่า 1,000 ม. และ อุณหภูมิต่ำกว่า 24 °C* พร้อมกัน